# Find, Load, and Visualise Earth Observation Imagery

This notebook demonstrates how to find, load, and visualise Earth observation imagery using cloud native approaches, which work well on your desktop or in cloud environments!

The notebook demonstrates how to find images from the Element 84 Earth Search STAC catalog, load them using `odc-stac`, and visualise them using extensions to xarray provided by `odc-geo`.

## Set up

The first step is to set up the required Python libraries and local imports.

* `odc.stac` and `pystac.client` are used to access the Earth Search STAC catalog
* `numpy` is used to manipulate data


In [ ]:
from odc.stac import configure_s3_access, load
from pystac.client import Client

The second step is to start a Dask client.

Dask supports local parallel processing and can help speed up computation times.

In [ ]:
from dask.distributed import Client as DaskClient

dask_client = DaskClient()
dask_client

## Part 1: Find

### 1.1 Connecting to the catalog

In [ ]:
# The catalog URL for the Earth Search STAC API
catalog = "https://earth-search.aws.element84.com/v1"

# Pystac_client's Client class is used to connect to the catalog
stac_client = Client.open(catalog)

# Configure settings for reading from Earth Search STAC
configure_s3_access(
    aws_unsigned=True,
)

### 1.2 Selecting an area to query

In [ ]:
from odc.geo import BoundingBox

# Nitmiluk National Park, Australia
bbox = BoundingBox(
    left=132.36,
    bottom=-14.28,
    right=132.44,
    top=-14.21,
    crs="EPSG:4326",
)

bbox.explore()

### 1.3 Set year and month to query

In [ ]:
# Set a date (year-month) for querying
date_query = "2026-09/2026-10"

### 1.4 Choose collections and filters

In [ ]:
# Set Earth Search product ID as the STAC "Collection"
collections_query = ["sentinel-2-l2a"]

# Set up a filter query. This is less than 20% cloud cover
filter_query = {"eo:cloud_cover": {"lt": 20}}

### 1.5 Running the query to indentify matching STAC items

In [ ]:
# Query with filtering for cloud cover
items = stac_client.search(
    collections=collections_query,
    intersects=bbox.polygon,
    datetime=date_query,
    query=filter_query,
).item_collection()

print(f"Found {len(items)} items within the bounding box, date range, and cloud cover filter.")

## Part 2: Load

### 2.1 Using odc-stac to load identified items

This may take a few minutes

In [ ]:
# Load our filtered data
ds_filtered = load(
    items,
    bands=["red", "green", "blue", "nir"],
    crs="utm",
    chunks={},
    resolution=10,
    groupby="solar_day",
    intersects=bbox.polygon,
).compute()

ds_filtered

### 2.2 Review loaded imagery

Identify which image you want to export and note the date.

In [ ]:
# to_array sets up a 3D array with the time dimension, which works directly
# with the plot function to make an RGB image
ds_filtered_rgb = ds_filtered[["red", "green", "blue"]]

ds_filtered_rgb.to_array().plot.imshow(col="time", col_wrap=2, robust=True, size=5)

## Part 3: Visualise

### 3.1 Select best image

Update the `best_image_date` parameter to match the date you identified in the previous step.

In [ ]:
best_image_date = "2026-09-12"

best_image = ds_filtered.sel(time=best_image_date).squeeze()

best_image

### 3.2 View the selected image on an interactive map

In [ ]:
best_image_rgb = best_image[["red", "green", "blue"]]

visualisation = best_image_rgb.odc.to_rgba()

visualisation.odc.explore()

### 3.3 Calculate and view NDVI

Calculate the normalised difference vegetation index (NDVI), which shows the riverine vegetation.
Storing each band as its own variable improves the readability of the code.

In [ ]:
nir = best_image.nir
red = best_image.red

best_image["ndvi"] = (nir - red) / (nir + red)

best_image

Display NDVI on the map.

In [ ]:
best_image["ndvi"].odc.explore(vmin=0, vmax=1)

### 3.4 Export to a cloud-optimised GeoTIFF

In [ ]:
best_image["ndvi"].odc.write_cog(f"sentinel2_ndvi_{best_image_date}.tif", overwrite=True)

## Part 4: Tidy up

In this section, we close the Dask client.
This prevents multiple clients being instantiated when using different notebooks.

In [ ]:
dask_client.close()

## Part 5: Comparing to Cogniscient

Cogniscient is an online tool that will allow you to do most of what we just did in your browser through a graphical user interface. 

To see the data we just loaded, go to [Cogniscient](https://cogniscient.auspatious.com/?cloud_cover_max=20&bbox=132.36_-14.28_132.44_-14.21&datetime=2026-09-12&width=858&preset=ndvi&viz_mode=index&vmax=1&colormap=viridis#12.0/-14.25/132.41)

For interest, Cogniscient saves its various settings in the URL for reproducibility. The URL for this example is:
https://cogniscient.auspatious.com/?cloud_cover_max=20&bbox=132.36_-14.28_132.44_-14.21&datetime=2026-09-12&width=858&preset=ndvi&viz_mode=index&vmax=1&colormap=viridis#12.0/-14.25/132.41

To learn more about using Cogniscient and generating STAC metadata for your images, see the [STAC_Creation_Example](./stac/STAC_Creation_Example.ipynb) notebook.